## Intro to POC Mode

[POC mode](https://nvflare.readthedocs.io/en/main/user_guide/poc_command.html) allows users to test the features of a full FLARE deployment on a single machine, without the overhead of a true distributed deployment and the need to establish secure communication between server and client systems.

Compared to the FL Simulator, where the job run is automated on a single system, POC mode allows you to establish and connect distinct server and client "systems" which can then be orchestrated using the FLARE Console.  This can be useful in preparation for a distributed deployment.

To get started, let's look at the NVFlare CLI usage for the poc subcommand:

In [1]:
!nvflare poc -h

usage: nvflare poc [-h] [-n [NUMBER_OF_CLIENTS]] [-p [PACKAGE]]
                   [-ex [EXCLUDE]] [-gpu [GPU [GPU ...]]] [--prepare]
                   [--start] [--stop] [--clean]

optional arguments:
  -h, --help            show this help message and exit
  -n [NUMBER_OF_CLIENTS], --number_of_clients [NUMBER_OF_CLIENTS]
                        number of sites or clients, default to 2
  -p [PACKAGE], --package [PACKAGE]
                        package directory, default to all = all packages, only
                        used for start/stop-poc commands when specified
  -ex [EXCLUDE], --exclude [EXCLUDE]
                        exclude package directory during --start or --stop,
                        default to , i.e. nothing to exclude
  -gpu [GPU [GPU ...]], --gpu [GPU [GPU ...]]
                        gpu device ids will be used as CUDA_VISIBLE_DEVICES.
                        used for poc start command
  --prepare             prepare poc workspace. export NVFLARE_HOME=<NVFLARE

### Preparing the POC environment
Before running POC mode, there are a couple important environment variables that should be set.

First, to simplify deploying the example apps in the NVFlare GitHub repo, you can set `NVFLARE_HOME` to the root of the GitHub clone.  In this case, we've cloned to our current working directory, so we can set it as:

In [2]:
import os
workdir=os.getcwd()
%env NVFLARE_HOME={workdir}/../NVFlare

env: NVFLARE_HOME=/flare/notebooks/../NVFlare


By default, POC mode uses a temporary workspace in /tmp/nvflare/poc.  We would like to keep the workspace within our working directory, so let's create a poc_workspace dir.  We can then use the `NVFLARE_POC_WORKSPACE` variable to define this as the POC workspace.

Note:  if you have previously created the poc_workspace, you will want to clean it up using the `nvflare poc --clean` command.

In [3]:
# !nvflare poc --clean
!mkdir poc_workspace
%env NVFLARE_POC_WORKSPACE={workdir}/poc_workspace

env: NVFLARE_POC_WORKSPACE=/flare/notebooks/poc_workspace


### Preparing the POC workspace

Now that we've configured out POC environment, we can prepare the POC workspace.  By default, this will generate POC packages for a server and two clients.

(Note that `nvflare poc --prepare` prompts you to create the workspace.)

In [4]:
!printf '%s\n' y | nvflare poc --prepare

prepare_poc at /flare/notebooks/poc_workspace for 2 clients
This will delete poc folder in /flare/notebooks/poc_workspace directory and create a new one. Is it OK to proceed? (y/N) Successfully creating poc folder at /flare/notebooks/poc_workspace.  Please read poc/Readme.rst for user guide.


******* Files generated by this poc command are NOT intended for production environments.
link examples from /flare/notebooks/../NVFlare/examples to /flare/notebooks/poc_workspace/admin/transfer


Let's take a look.

In [5]:
!tree poc_workspace

poc_workspace
├── Readme.rst
├── admin
│   ├── local
│   ├── startup
│   │   ├── fed_admin.json
│   │   └── fl_admin.sh
│   └── transfer -> /flare/notebooks/../NVFlare/examples
├── server
│   ├── local
│   │   ├── log.config
│   │   └── resources.json
│   └── startup
│       ├── fed_server.json
│       ├── start.sh
│       ├── stop_fl.sh
│       └── sub_start.sh
├── site-1
│   ├── local
│   │   ├── log.config
│   │   └── resources.json
│   └── startup
│       ├── fed_client.json
│       ├── start.sh
│       ├── stop_fl.sh
│       └── sub_start.sh
└── site-2
    ├── local
    │   ├── log.config
    │   └── resources.json
    └── startup
        ├── fed_client.json
        ├── start.sh
        ├── stop_fl.sh
        └── sub_start.sh

13 directories, 21 files


### Running the POC Deployment

When starting the POC deployment, it's necessary to use a separate terminal since the `nvflare poc --start` command will run  in the foreground emitting output from the server and any connected clients.

Also note that `nvflare poc --start` starts all participants, including the admin console.  It's often nice to start server and clients separately so that we can interact with the deployment using a separate admin console.  To do this, we'll pass the `-ex admin` arg to exclude the admin client from the initial POC run and use the FLARE API to run admin commands separately.

So pop open the launcher, launch a terminal, and run (remembering to set the NVFLARE_POC_WORKSPACE and NVFLARE_HOME vars!):

```shell
export NVFLARE_POC_WORKSPACE=$(pwd -P)/notebooks/poc_workspace
export NVFLARE_HOME=$(pwd -P)/NVFlare
nvflare poc --start -ex admin
```

Keep this terminal open so you can continue to watch server and client output.

### Using the FLARE API to connect to the POC deployment

The admin directory contains the startup script for the FLARE Console, which can be used interactively to operate a running FLARE deployment.  A FLARE deployment can also be managed using the FLARE API, which will use the configuration in the admin directory to connect to the FLARE server.  Since we already have the server and clients running in the background from the above terminal commands, we'll use FLARE API to start a new admin session and connect.

To get started, we need to import the FLARE API class and initialize session.

In [6]:
from nvflare.fuel.flare_api.flare_api import new_insecure_session

admin_session = new_insecure_session(startup_kit_location = workdir + "/poc_workspace/admin")
print(admin_session.get_system_info())

SystemInfo
server_info:
status: stopped, start_time: Thu Mar  9 17:08:02 2023
client_info:
site-1(last_connect_time: Thu Mar  9 17:08:07 2023)
site-2(last_connect_time: Thu Mar  9 17:08:10 2023)
job_info:



### Launching a job with the FLARE API
Next we can use the FLARE API to launch one of the hello-world examples and monitor its status.  Note the difference here as compared to the Simulator example.  Because we're running a POC deployment with unique workspaces for the FLARE server and clients, we don't need to define a local workspace for job results before submitting the job.

All that's required to launch the job is the path to the job configuration.  This configuration is pushed to the server workspace and deployed to clients, and all job results are collected back in the server workspace.  After the job completes, we can use the FLARE API to download the results of the job from the server workspace to our admin directory.


In [9]:
!tree /flare/NVFlare/examples/hello-world/hello-pt/jobs/hello-pt

/flare/NVFlare/examples/hello-world/hello-pt/jobs/hello-pt
├── app
│   ├── config
│   │   ├── config_fed_client.json
│   │   └── config_fed_server.json
│   └── custom
│       ├── cifar10trainer.py
│       ├── cifar10validator.py
│       ├── pt_constants.py
│       ├── pt_model_locator.py
│       ├── simple_network.py
│       └── test_custom.py
└── meta.json

3 directories, 9 files


In [10]:
path_to_job_config = "/flare/NVFlare/examples/hello-world/hello-pt/jobs/hello-pt"
job_id = admin_session.submit_job(path_to_job_config)
print("Submitted job with job ID" + job_id)

### Monitoring the state of the FLARE deployment and job status

Now that the job is submitted, we can use the FLARE API to query the state of the system, show job status, and display job metadata.  These capabilities are especially useful through the course of a FLARE experiment, when you typically execute multiple jobs through the course of the course of the experiment.

For example, you can query the state of all jobs (with optional detailed output including job metadata), or query the metadata for a specific job by ID.

In [13]:
import json

# Job Status
jobs_output = admin_session.list_jobs()
jobs_detail = admin_session.list_jobs(detailed=True)
print("Job Status")
print(json.dumps((jobs_output), indent=2))
print("\nJob Detail")
print(json.dumps((jobs_detail), indent=2))

# Job Metadata
print("\nJob Metadata")
admin_session.get_job_meta(job_id)

Job Status
[
  {
    "job_id": "86b5e7e3-69b1-4d2c-a0f0-643e9fbca643",
    "job_name": "hello-pt",
    "status": "RUNNING",
    "submit_time": "2023-03-09T17:08:59.671175+00:00",
    "duration": "0:00:53.236106"
  },
  {
    "job_id": "426cfd61-409f-438b-8efd-5eb6517e3f65",
    "job_name": "hello-pt",
    "status": "FINISHED:COMPLETED",
    "submit_time": "2023-03-08T19:39:16.001409+00:00",
    "duration": "0:10:15.617658"
  }
]

Job Detail
[
  {
    "name": "hello-pt",
    "resource_spec": {},
    "min_clients": 2,
    "deploy_map": {
      "app": [
        "@ALL"
      ]
    },
    "job_folder_name": "hello-pt",
    "submitter_name": "admin",
    "submitter_org": "global",
    "submitter_role": "super",
    "job_id": "86b5e7e3-69b1-4d2c-a0f0-643e9fbca643",
    "submit_time": 1678381739.671175,
    "submit_time_iso": "2023-03-09T17:08:59.671175+00:00",
    "start_time": "2023-03-09 17:09:00.614052",
    "duration": "0:00:53.240159",
    "status": "RUNNING",
    "job_deploy_detail": [


{'name': 'hello-pt',
 'resource_spec': {},
 'min_clients': 2,
 'deploy_map': {'app': ['@ALL']},
 'job_folder_name': 'hello-pt',
 'submitter_name': 'admin',
 'submitter_org': 'global',
 'submitter_role': 'super',
 'job_id': '86b5e7e3-69b1-4d2c-a0f0-643e9fbca643',
 'submit_time': 1678381739.671175,
 'submit_time_iso': '2023-03-09T17:08:59.671175+00:00',
 'start_time': '2023-03-09 17:09:00.614052',
 'duration': 'N/A',
 'status': 'RUNNING',
 'job_deploy_detail': ['server: OK', 'site-1: OK', 'site-2: OK'],
 'schedule_count': 1,
 'last_schedule_time': 1678381740.5233593,
 'schedule_history': ['2023-03-09 17:09:00: scheduled']}

### Monitoring a job run with a callback function
You can also construct a simple callback function to monitor job status during a run.

In [26]:
from nvflare.fuel.flare_api.flare_api import Session

def sample_cb(
        session: Session, job_id: str, job_meta, *cb_args, **cb_kwargs
    ) -> bool:
    if job_meta["status"] == "RUNNING":
        if cb_kwargs["cb_run_counter"]["count"] < 3:
            print(job_meta)
            print(cb_kwargs["cb_run_counter"])
        else:
            print(".", end="")
    else:
        print("\n" + str(job_meta))
    
    cb_kwargs["cb_run_counter"]["count"] += 1
    return True

admin_session.monitor_job(job_id, cb=sample_cb, cb_run_counter={"count":0})


{'name': 'hello-pt', 'resource_spec': {}, 'min_clients': 2, 'deploy_map': {'app': ['@ALL']}, 'job_folder_name': 'hello-pt', 'submitter_name': 'admin', 'submitter_org': 'global', 'submitter_role': 'super', 'job_id': '86b5e7e3-69b1-4d2c-a0f0-643e9fbca643', 'submit_time': 1678381739.671175, 'submit_time_iso': '2023-03-09T17:08:59.671175+00:00', 'start_time': '2023-03-09 17:09:00.614052', 'duration': '0:03:13.037359', 'status': 'FINISHED:COMPLETED', 'job_deploy_detail': ['server: OK', 'site-1: OK', 'site-2: OK'], 'schedule_count': 1, 'last_schedule_time': 1678381740.5233593, 'schedule_history': ['2023-03-09 17:09:00: scheduled']}


<MonitorReturnCode.JOB_FINISHED: 0>

### Stopping the POC deployment (important!)
Once the job has completed, we can stop the server and clients in the POC deployent.  This is necessary to free up ports for the following notebook examples!

In [ ]:
!nvflare poc --stop

### Retrieving job results
When the cell above monitoring job output shows that the job has finished with `<MonitorReturnCode.JOB_FINISHED: 0>`, we can use the FLARE API to download job results.  This is useful when running the FLARE API on a remote deployment and you wish to review job artifacts on your local laptop or workstation.

In [27]:
job_download = admin_session.download_job_result(job_id)
print("Job download path: " + job_download)

Job download path: /flare/notebooks/poc_workspace/admin/transfer/86b5e7e3-69b1-4d2c-a0f0-643e9fbca643


In [28]:
!tree {job_download}

/flare/notebooks/poc_workspace/admin/transfer/86b5e7e3-69b1-4d2c-a0f0-643e9fbca643
├── job
│   └── hello-pt
│       ├── app
│       │   ├── config
│       │   │   ├── config_fed_client.json
│       │   │   └── config_fed_server.json
│       │   └── custom
│       │       ├── cifar10trainer.py
│       │       ├── cifar10validator.py
│       │       ├── pt_constants.py
│       │       ├── pt_model_locator.py
│       │       ├── simple_network.py
│       │       └── test_custom.py
│       └── meta.json
└── workspace
    ├── app_server
    │   ├── FL_global_model.pt
    │   ├── config
    │   │   ├── config_fed_client.json
    │   │   └── config_fed_server.json
    │   └── custom
    │       ├── __pycache__
    │       │   ├── pt_constants.cpython-38.pyc
    │       │   ├── pt_model_locator.cpython-38.pyc
    │       │   └── simple_network.cpython-38.pyc
    │       ├── cifar10trainer.py
    │       ├── cifar10validator.py
    │       ├── pt_constants.py
    │       ├── pt_model_locator.py

In [29]:
!tail {job_download}/workspace/log.txt

2023-03-09 17:12:00,863 - ServerRunner - INFO - [identity=example_project, run=86b5e7e3-69b1-4d2c-a0f0-643e9fbca643, wf=cross_site_validate, peer=site-2, peer_run=86b5e7e3-69b1-4d2c-a0f0-643e9fbca643]: server runner is finalizing - asked client to end the run
2023-03-09 17:12:01,479 - ServerRunner - INFO - [identity=example_project, run=86b5e7e3-69b1-4d2c-a0f0-643e9fbca643, wf=cross_site_validate]: ABOUT_TO_END_RUN fired
2023-03-09 17:12:08,473 - ServerRunner - INFO - [identity=example_project, run=86b5e7e3-69b1-4d2c-a0f0-643e9fbca643, wf=cross_site_validate]: END_RUN fired
2023-03-09 17:12:08,474 - ServerEngine - INFO - Start saving snapshot on server.
2023-03-09 17:12:08,480 - ServerEngine - INFO - The snapshot: /tmp/nvflare/snapshot-storage/86b5e7e3-69b1-4d2c-a0f0-643e9fbca643 has been removed.
2023-03-09 17:12:08,480 - ServerRunner - INFO - [identity=example_project, run=86b5e7e3-69b1-4d2c-a0f0-643e9fbca643, wf=cross_site_validate]: Server runner finished.
2023-03-09 17:12:09,448 -

In [30]:
cross_val_file = open(job_download + "/workspace/cross_site_val/cross_val_results.json")
cross_val_json = json.load(cross_val_file)
print(json.dumps(cross_val_json, indent=2))

{
  "site-2": {
    "SRV_server": {
      "val_acc": 0.2577
    },
    "site-1": {
      "val_acc": 0.2747
    },
    "site-2": {
      "val_acc": 0.281
    }
  },
  "site-1": {
    "SRV_server": {
      "val_acc": 0.2577
    },
    "site-1": {
      "val_acc": 0.2747
    },
    "site-2": {
      "val_acc": 0.281
    }
  }
}
